# Composable Stopping Criteria


Stopping criteria can be composed with the `|` operator. `minimize()` stops when **any** criterion triggers. The reason is recorded in `result.status`.


In [ ]:
import numpy as np
from optiland import optic
from optiland.optimization import minimize, OptimizationProblem
from optiland.optimization.stopping.criteria import MaxIter, CostTolerance, GradNormTolerance

lens = optic.Optic()
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(index=1, thickness=7, radius=50, material="N-BK7", is_stop=True)
lens.surfaces.add(index=2, thickness=45, radius=-50)
lens.surfaces.add(index=3)
lens.set_aperture(aperture_type="EPD", value=10)
lens.fields.set_type("angle")
lens.fields.add(y=0.0)
lens.fields.add(y=0.7)
lens.wavelengths.add(value=0.5876, is_primary=True)
lens.update_paraxial()


In [ ]:
problem = OptimizationProblem()
for field in lens.fields.get_field_coords():
    input_data = {"optic": lens, "surface_number": -1, "Hx": field[0], "Hy": field[1],
                  "num_rays": 5, "wavelength": 0.5876, "distribution": "hexapolar"}
    problem.add_operand("rms_spot_size", target=0, weight=1, input_data=input_data)
problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)
problem.add_variable(lens, "thickness", surface_number=2)


Build a custom stopping criterion: stop after 50 iterations **or** when the relative merit change falls below 1e-5.


In [ ]:
stop = MaxIter(50) | CostTolerance(1e-5)
result = minimize(problem, "dls", stop=stop)
print(result)
print(f"Stopped because: {result.status}")


You can also chain three or more criteria: `MaxIter(100) | CostTolerance(1e-6) | GradNormTolerance(1e-4)`.


In [ ]:
import copy
lens2 = copy.deepcopy(lens)
problem2 = OptimizationProblem()
for field in lens2.fields.get_field_coords():
    input_data = {"optic": lens2, "surface_number": -1, "Hx": field[0], "Hy": field[1],
                  "num_rays": 5, "wavelength": 0.5876, "distribution": "hexapolar"}
    problem2.add_operand("rms_spot_size", target=0, weight=1, input_data=input_data)
problem2.add_variable(lens2, "radius", surface_number=1)
problem2.add_variable(lens2, "radius", surface_number=2)
problem2.add_variable(lens2, "thickness", surface_number=2)
stop2 = MaxIter(100) | CostTolerance(1e-6) | GradNormTolerance(1e-4)
result2 = minimize(problem2, "dls", stop=stop2)
print(result2)
